In [14]:
import pandas as pd

In [15]:
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # remove leading and trailing whitespace from all string columns
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip()

    # convert to lowercase
    df.columns = df.columns.str.lower()

    # remove punctuation
    df.columns = df.columns.str.replace(r'[^\w]', '', regex=True)

    # return all columns that have a Dtype of numeric
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

    # convert numeric columns to appropriate types, coercing errors to NaN
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

    # replace multiple spaces with a single space
    df = df.replace(r'\s+', ' ', regex=True)

    # return all columns that have a Dtype of numeric
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

    # convert numeric columns to appropriate types, coercing errors to NaN
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

In [16]:
def mask_df(df: pd.DataFrame) -> pd.DataFrame:
    # filter the dataframe to only include rows where:
    mask = (
          (df["ship_plant_code"] == 512) # solo planta CUU 512
        & (df["u_volumen"] >= 0.5)
        & (df["u_volumen"] <= 7.5) # entre volumenes de 0.5 a 7.5
        & (df["u_cicle"] >= 50)
        & (df["u_cicle"] <= 110) # entre tiempos de ciclo de 50 a 110
    )

    df = df.loc[mask].copy()

    return df

In [17]:
df = pd.read_csv("../../data/01-raw/2026.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8389 entries, 0 to 8388
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Año                  8389 non-null   int64  
 1   tkt_code             8389 non-null   int64  
 2   order_date           8389 non-null   str    
 3   order_code           8389 non-null   int64  
 4   start_time           8389 non-null   str    
 5   truck_code           8389 non-null   int64  
 6   ship_plant_code      8389 non-null   int64  
 7   u_Volumen            8389 non-null   float64
 8   typed_time           8389 non-null   str    
 9   at_plant_time        8125 non-null   str    
 10  u_Cicle              8125 non-null   float64
 11  name                 8389 non-null   str    
 12  Nombre del proyecto  8389 non-null   str    
 13  ship_addr_line       8389 non-null   str    
dtypes: float64(2), int64(5), str(7)
memory usage: 917.7 KB


In [18]:
# iterate through files in the 01-raw directory and read them into a single dataframe, cleaning and masking each file before concatenating them together
import os
raw_dir = "../../data/01-raw"
dfs = []
for filename in os.listdir(raw_dir):
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join(raw_dir, filename))
        df = clean_df(df)
        df = mask_df(df)
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35335 entries, 0 to 35334
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   año                35335 non-null  int64  
 1   tkt_code           35335 non-null  int64  
 2   order_date         35335 non-null  str    
 3   order_code         35335 non-null  int64  
 4   start_time         35335 non-null  str    
 5   truck_code         35335 non-null  float64
 6   ship_plant_code    35335 non-null  int64  
 7   u_volumen          35335 non-null  float64
 8   typed_time         35335 non-null  str    
 9   at_plant_time      35335 non-null  str    
 10  u_cicle            35335 non-null  float64
 11  name               35335 non-null  str    
 12  nombredelproyecto  35335 non-null  str    
 13  ship_addr_line     35324 non-null  str    
dtypes: float64(3), int64(4), str(7)
memory usage: 3.8 MB


In [19]:
df["ship_plant_code"].value_counts()

ship_plant_code
512    35335
Name: count, dtype: int64

In [20]:
df.to_csv("../../data/02-cleaned/cleaned_data.csv", index=False)